# **Introduction**

This notebook is a baseline for training a simple neuronal network on GPU without augmentations and should be easy to understand for anyone. In the Session options -> Accelerator ->  should select a GPU

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# **Import packages**

In [ ]:
import torch 
import torch.nn as nn 
import matplotlib.pyplot as plt # visualization
from torch.utils.data import DataLoader, TensorDataset # load the data
from sklearn.model_selection import StratifiedKFold
from torch.optim import SGD
from tqdm import tqdm # track the progress 

# **Import training data from Kaggle**

In [ ]:
train_data=pd.read_csv("/kaggle/input/digit-recognizer/train.csv")

Let's look at the first 5 rows:

In [ ]:
train_data.head()

The dataset contains the label (0 to 9) in the first column and then either a value between 0 and 255 for each of the 784 pixels. The higher the number, the darker the color. We will have a look at the first picture:

In [ ]:
# Take first row [0], leave out the label column [1:]
x1=train_data.iloc[0,1:].values

In [ ]:
# For visualization of the image, reshape the first into a 28 x 28 shape:
x1re=x1.reshape(28,28)

In [ ]:
# Plot the reshaped image
plt.imshow(x1re)

**Split the training data into training and validation set**

In [ ]:
# Generate a config class:
class CFG:
    batch_size = 32 # Specifies how many images are in one batch
    lr = 1e-3 # Learning rate, how much the model adapts the parameters in each step
    num_epochs = 200 # Number of epochs, how long the model is training
    n_folds = 5 # Number of folds

In [ ]:
x = train_data
y = train_data['label']
cv = StratifiedKFold(n_splits=CFG.n_folds) # Stratified k fold preserves number of occurences in dataset
split = list(cv.split(x,y)) # generates a list of the train and validation split for each fold

In [ ]:
# Add a fold column into the train dataset
train_data['fold'] = -1 #dummy which gets overwriten in next 2 lines
for fold,(tr_idx, val_idx) in enumerate(split):
    train_data.iloc[val_idx,-1] = fold 
# loops over the just generated split list with training and validation index and adds this into the fold column

Here us what the training data looks like now:

In [ ]:
train_data.head()

In [ ]:
# Are the folds the same size?
train_data['fold'].value_counts(normalize=True)

Yes, the data is evenly split into 5 folds, each containing 20% of the data.

In [ ]:
# How is the distribution of the lables in the dataset?
train_data['label'].value_counts(normalize=True).sort_index()

In [ ]:
plt.hist(train_data['label'])

Similar counts for all digits.

**Prepare the dataloader of training and validation set**

In [ ]:
def prepare_loaders(fold, batch_size):
    train_df = train_data[train_data["fold"]!=fold].reset_index(drop=True)
    valid_df = train_data[train_data["fold"]==fold].reset_index(drop=True)

    X = train_df.iloc[:,1:-1].values / 255 # norm picture to values between 0-1
    y = train_df.iloc[:,0].values
    train_dataset = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))

    X_val = valid_df.iloc[:,1:-1].values / 255 # norm picture to values between 0-1
    y_val = valid_df.iloc[:,0].values
    valid_dataset = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True, drop_last=False)
    valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)
    
    return train_loader, valid_loader

Load the training and validation loader for fold 0.

In [ ]:
train_loader, valid_loader = prepare_loaders(fold=0, batch_size=CFG.batch_size) 

In [ ]:
model = nn.Sequential(nn.Linear(784, 1024), # 1st linear layer
                      nn.LeakyReLU(negative_slope=0.05), # Leaky rectified linear unit 
                      nn.Linear(1024, 512), # 2nd linear layer
                      nn.LeakyReLU(negative_slope=0.05),
                      nn.Linear(512, 256), # 3rd linear layer
                      nn.LeakyReLU(negative_slope=0.05),
                      nn.Linear(256, 10)) # 4th linear layer outputs 10 features
model.cuda() # Push the model to GPU
optimizer = SGD(model.parameters(), lr=CFG.lr) # Stochastic gradient descent
criterion = nn.CrossEntropyLoss() #loss function

# Generate the NN model
Initiate a very simple NN with 4 linear layers.

Train the model and calculate validation loss (Cross Entropy Loss) and accuracy. 
Save the model parameter (later loaded in inference notebook). 

In [ ]:
train_loss = []
v_loss2 = []
acc2 = []
for i in range(CFG.num_epochs):
    model.train()
    for data in tqdm(train_loader):
        # Set the gradients to zero
        optimizer.zero_grad()
        # Run a forward pass
        feature, target = data
        prediction = model(feature.float().cuda())
        prediction = prediction.cpu()
        # Compute the loss
        loss = criterion(prediction, target)
        # Compute the gradients
        loss.backward()
        # Update the model's parameters
        optimizer.step()
        train_loss += [loss.item()]
    val_loss = []
    accuracy = []
    model.eval()
    with torch.no_grad():
        for data in tqdm(valid_loader):
            feature, target = data
            prediction = model(feature.float().cuda())
            prediction = prediction.cpu()
            loss = criterion(prediction, target)
            val_loss += [loss.item()]
            accuracy += [(prediction.softmax(1).argmax(1) == target).float().mean()]
    v_loss = np.array(val_loss).mean()
    print(f'Epoch: {i}')
    print(f'Val_loss: {v_loss}')
    acc = np.array(accuracy).mean()
    print(f'Accuracy: {acc}')
    v_loss2 += [v_loss]
    acc2 += [acc]
    PATH = f"last_epoch-{fold:02d}.bin"
    torch.save(model.state_dict(), PATH) # Saves the model weights

**Plot the training and validation loss**

In [ ]:
all_loss = np.array(train_loss)

In [ ]:
plt.plot(all_loss)

In [ ]:
plt.plot(v_loss2)

**Plot the accuracy**

In [ ]:
plt.plot(acc2)

This notebook just trains a model, in order to create a submission to the competition I generated an inference notebook where the trained model / weights are loaded. In this case the same notebook could train the model and predict the unknown labels, as the computational costs are low. However, if the model is big, just importing the weights into a separate [Inference notebook](https://www.kaggle.com/code/bridgeoverwater/gpu-infer-digit-recognizer) is more efficient.


[](http://)